In [1]:
def fix_untrained_tokens(model, eps = 1e-16):
    """
    Llama-3 for eg has untrained vectors in the base model.
    These include <|eot_id|>, <|start_header_id|>, <|end_header_id|>
    We reset them to the mean of the rest of the tokens
    """
    embedding_matrix = model.get_input_embeddings ().weight.data
    lm_head_matrix   = model.get_output_embeddings().weight.data

    # Get untrained tokens
    indicator_untrained = torch.amax(embedding_matrix, axis = 1) <= eps
    where_untrained = torch.where(indicator_untrained)[0]
    n_untrained = where_untrained.shape[0]
    n_trained = embedding_matrix.shape[0] - n_untrained
    if n_untrained != 0:
        print(
            f"Unsloth: Not an error, but your model has {n_untrained} untrained tokens.\n"\
            "We shall set them to the mean of the other trained tokens."
        )
    pass

    # First set untrained to all 0s - sometimes it's not! 1e-23 for bfloat16
    embedding_matrix[where_untrained] = 0
    lm_head_matrix  [where_untrained] = 0

    # Find sum
    sum_embedding  = torch.sum(embedding_matrix, dtype = torch.float32, axis = 0)
    sum_lm_head    = torch.sum(lm_head_matrix,   dtype = torch.float32, axis = 0)

    # Find correct average by dividing by sum of trained tokens
    mean_embedding = (sum_embedding / n_trained).to(embedding_matrix.dtype)
    mean_lm_head   = (sum_lm_head   / n_trained).to(lm_head_matrix  .dtype)

    # Set them to the mean
    embedding_matrix[where_untrained] = mean_embedding
    lm_head_matrix  [where_untrained] = mean_lm_head

    return mean_embedding, mean_lm_head

In [4]:
import torch
import transformers


In [5]:
# model = transformers.AutoModelForCausalLM.from_pretrained(
#         "meta-llama/Meta-Llama-3.1-8B-Instruct",
#         trust_remote_code=True,
#         attn_implementation="flash_attention_2",
#         torch_dtype=torch.bfloat16,
#     )
#     # Tie the weights - Commonly used for memory efficient training?
# model.tie_weights()

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00,  4.27it/s]


In [6]:
embedding_matrix = model.get_input_embeddings ().weight.data
lm_head_matrix   = model.get_output_embeddings().weight.data

    # Get untrained tokens
indicator_untrained = torch.amax(embedding_matrix, axis = 1) <= 1e-16
    

In [7]:
indicator_untrained

tensor([False, False, False,  ...,  True,  True,  True])

In [8]:
where_untrained = torch.where(indicator_untrained)[0]
n_untrained = where_untrained.shape[0]
n_trained = embedding_matrix.shape[0] - n_untrained

In [9]:
n_untrained

266

In [10]:
n_trained

127990

In [11]:
where_untrained

tensor([   124,    125,    177,    178,    179,    180,    181,    182,    183,
           184,    185,    186,    187, 100617, 103003, 106710, 122746, 128002,
        128003, 128004, 128005, 128011, 128012, 128013, 128014, 128015, 128016,
        128017, 128018, 128019, 128020, 128021, 128022, 128023, 128024, 128025,
        128026, 128027, 128028, 128029, 128030, 128031, 128032, 128033, 128034,
        128035, 128036, 128037, 128038, 128039, 128040, 128041, 128042, 128043,
        128044, 128045, 128046, 128047, 128048, 128049, 128050, 128051, 128052,
        128053, 128054, 128055, 128056, 128057, 128058, 128059, 128060, 128061,
        128062, 128063, 128064, 128065, 128066, 128067, 128068, 128069, 128070,
        128071, 128072, 128073, 128074, 128075, 128076, 128077, 128078, 128079,
        128080, 128081, 128082, 128083, 128084, 128085, 128086, 128087, 128088,
        128089, 128090, 128091, 128092, 128093, 128094, 128095, 128096, 128097,
        128098, 128099, 128100, 128101, 

In [12]:
embedding_matrix.shape

torch.Size([128256, 4096])

In [13]:
lm_head_matrix   = model.get_output_embeddings().weight.data

In [14]:
lm_head_matrix.shape

torch.Size([128256, 4096])

In [ ]:
model.set_input_embeddings